In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
import json

from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.metrics import precision_score, recall_score, f1_score

from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
torch.backends.cudnn.benchmark = True

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset"

TRAIN_IMG = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/train/image"
TRAIN_ANNO = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/train/annos"

VAL_IMG = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/validation/image"
VAL_ANNO = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/validation/annos"

# print(len(os.listdir(IMAGE_DIR)))

In [ ]:
train_images = sorted(os.listdir(TRAIN_IMG))[:30000]
val_images = sorted(os.listdir(VAL_IMG))[:5000]

print(len(train_images), len(val_images))

In [ ]:
NUM_CLASSES = 6
category_map = {1:1, 2:2, 7:3, 8:4, 9:5}

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

In [ ]:
def get_train_transform():

    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.RandomRotate90(p=0.5),

            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.1,
                rotate_limit=15,
                border_mode=cv2.BORDER_CONSTANT,
                p=0.5
            ),

            A.RandomBrightnessContrast(p=0.5),

            A.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1,
                p=0.5
            ),

            A.GaussianBlur(p=0.2),

            A.Resize(384, 384),
            A.Normalize(),

            ToTensorV2()

        ],

        bbox_params=A.BboxParams(
            format='pascal_voc',
            label_fields=['labels'],
            min_area=1,
            min_visibility=0.3,
            clip=True,
            filter_invalid_bboxes=True
        )
    )

In [ ]:
def get_val_transform():

    return A.Compose(
        [
            A.Resize(384, 384),
            A.Normalize(),
            ToTensorV2()
        ],

        bbox_params=A.BboxParams(
            format='pascal_voc',
            label_fields=['labels'],
            min_area=1,
            min_visibility=0.3,
            clip=True,
            filter_invalid_bboxes=True
        )
    )

In [ ]:
class ClothingDataset(Dataset):

    def __init__(self, image_dir, mask_dir, image_list=None, transforms=None):

        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transforms = transforms
    
        if image_list is None:
            self.images = sorted(os.listdir(image_dir))
        else:
            self.images = image_list

    def __getitem__(self, idx):
    
        img_name = self.images[idx]
    
        img_path = os.path.join(self.image_dir, img_name)
    
        json_name = os.path.splitext(img_name)[0] + ".json"
        json_path = os.path.join(self.mask_dir, json_name)
    
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
        h, w = img.shape[:2]
    
        with open(json_path) as f:
            anno = json.load(f)
    
        masks = []
        boxes = []
        labels = []
    
        for item in anno.values():

            cat = item["category_id"]

            # skip categories not used
            if cat not in category_map:
                continue

            label = category_map[cat]

            # bounding box
            x1, y1, x2, y2 = item["bounding_box"]

            # segmentation polygons → mask
            mask = np.zeros((h, w), dtype=np.uint8)

            for poly in item["segmentation"]:
                poly = np.array(poly).reshape(-1,2).astype(np.int32)
                cv2.fillPoly(mask, [poly], 1)

            masks.append(mask)
            boxes.append([x1,y1,x2,y2])
            labels.append(label)

        # handle images with no valid objects
        if len(masks) == 0:
            return self.__getitem__((idx+1) % len(self.images))

        masks = np.array(masks)

        if self.transforms:

            transformed = self.transforms(
                image=img,
                masks=list(masks),
                bboxes=boxes,
                labels=labels
            )

            img = transformed["image"]
            masks = transformed["masks"]
            boxes = transformed["bboxes"]
            labels = transformed["labels"]

            masks = torch.stack([torch.as_tensor(m) for m in masks])
            boxes = torch.as_tensor(boxes,dtype=torch.float32)

            # Fix for single box case
            if boxes.ndim == 1:
                boxes = boxes.unsqueeze(0)
            labels = torch.as_tensor(labels,dtype=torch.int64)

        else:

            img = torch.as_tensor(img,dtype=torch.float32).permute(2,0,1)/255.0
            masks = torch.as_tensor(masks,dtype=torch.uint8)
            boxes = torch.as_tensor(boxes,dtype=torch.float32)

            # Fix for single box case
            if boxes.ndim == 1:
                boxes = boxes.unsqueeze(0)
            labels = torch.as_tensor(labels,dtype=torch.int64)

        image_id = torch.tensor([idx])

        if boxes.shape[0] == 0:
            return self.__getitem__((idx+1) % len(self.images))

        area = (boxes[:,3]-boxes[:,1])*(boxes[:,2]-boxes[:,0])

        iscrowd = torch.zeros((len(labels),),dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        return img, target

    def __len__(self):
        return len(self.images)

In [ ]:
train_dataset = ClothingDataset(
    TRAIN_IMG,
    TRAIN_ANNO,
    image_list=train_images,
    transforms=get_train_transform()
)

val_dataset = ClothingDataset(
    VAL_IMG,
    VAL_ANNO,
    image_list=val_images,
    transforms=get_val_transform()
)

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

In [ ]:
def compute_iou(pred_mask, true_mask):

    intersection = (pred_mask & true_mask).sum()
    union = (pred_mask | true_mask).sum()

    if union == 0:
        return 0.0

    return intersection / union


def compute_dice(pred_mask, true_mask):

    intersection = (pred_mask & true_mask).sum()

    return (2 * intersection) / (pred_mask.sum() + true_mask.sum() + 1e-6)

In [ ]:
def evaluate_metrics(model, loader):

    model.eval()

    map_metric = MeanAveragePrecision()

    all_pred_labels = []
    all_true_labels = []

    iou_scores = []
    dice_scores = []

    with torch.no_grad():

        for images, targets in loader:

            images = [img.to(device) for img in images]
            targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

            outputs = model(images)

            map_metric.update(outputs, targets)

            for i in range(len(outputs)):

                pred = outputs[i]
                tgt = targets[i]

                pred_labels = pred["labels"].cpu().numpy()
                true_labels = tgt["labels"].cpu().numpy()

                all_pred_labels.extend(pred_labels)
                all_true_labels.extend(true_labels)

                if len(pred["masks"]) > 0:

                    pred_masks = (pred["masks"] > 0.5).squeeze(1).cpu().numpy()
                    true_masks = tgt["masks"].cpu().numpy()

                    for pm, tm in zip(pred_masks, true_masks):

                        iou_scores.append(compute_iou(pm, tm))
                        dice_scores.append(compute_dice(pm, tm))

    map_results = map_metric.compute()

    precision = 0
    recall = 0
    f1 = 0

    mean_iou = np.mean(iou_scores) if len(iou_scores) > 0 else 0
    mean_dice = np.mean(dice_scores) if len(dice_scores) > 0 else 0

    return {
        "mAP": map_results["map"].item(),
        "mean_iou": mean_iou,
        "dice": mean_dice
    }

In [ ]:
def get_model_transfer(num_classes):

    model = maskrcnn_resnet50_fpn(weights="DEFAULT")

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features,
        num_classes
    )

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels

    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        256,
        num_classes
    )

    model.rpn.pre_nms_top_n_train = 1500
    model.rpn.post_nms_top_n_train = 800

    model.rpn.pre_nms_top_n_test = 1500
    model.rpn.post_nms_top_n_test = 800

    model.roi_heads.detections_per_img = 75

    return model

In [ ]:
model = get_model_transfer(NUM_CLASSES)
model.to(device)

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

# optimizer = torch.optim.AdamW(params, lr=1e-4) #early training
optimizer = torch.optim.AdamW(params, lr=5e-5) #later training

# lr_scheduler = torch.optim.lr_scheduler.StepLR(
#     optimizer,
#     step_size=3,
#     gamma=0.1
# ) # Early training

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
) #late training

In [ ]:
scaler = torch.cuda.amp.GradScaler()
def train_one_epoch(model, loader, optimizer, scaler):

    model.train()

    total_loss = 0

    for images, targets in tqdm(loader):

        images = [img.to(device) for img in images]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

        with torch.cuda.amp.autocast():
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += losses.item()

    return total_loss / len(loader)

In [ ]:
# ==============================
# Paths (Kaggle safe)
# ==============================
WORKING_CHECKPOINT = "/kaggle/working/best_maskrcnn_transfer.pth"
INPUT_CHECKPOINT = "/kaggle/input/datasets/vissshhhwa/best-mask-rcnn/best_maskrcnn_transfer.pth"
LOG_PATH = "/kaggle/working/training_log.txt"

# ==============================
# Initialize
# ==============================
best_map = 0
best_epoch = 0
start_epoch = 0
checkpoint=None
EPOCHS = 15

# ==============================
# Resume from checkpoint (if exists)
# ==============================
if os.path.exists(WORKING_CHECKPOINT):
    print("Resuming from WORKING checkpoint...")

    checkpoint = torch.load(WORKING_CHECKPOINT, map_location=device)
    
elif os.path.exists(INPUT_CHECKPOINT):
    print("Loading checkpoint...")

    checkpoint = torch.load(INPUT_CHECKPOINT, map_location=device)

if checkpoint is not None:
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    start_epoch = checkpoint.get('epoch', 0)
    best_map = checkpoint.get('mAP', 0)

    print(f"Start Epoch: {start_epoch}, Best mAP: {best_map}")

# ==============================
# Initialize log file
# ==============================
with open(LOG_PATH, "a") as f:
    f.write("\n=============================\n")
    f.write("New Training Session\n")
    f.write("=============================\n")

# ==============================
# Training Loop
# ==============================
for epoch in range(start_epoch, EPOCHS):

    print(f"\nEpoch {epoch+1}")
    print("----------------------------")

    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)

    metrics = evaluate_metrics(model, val_loader)
    current_map = metrics["mAP"]

    print("Train Loss:", train_loss)

    print("\nDetection Metrics")
    print("mAP:", current_map)

    print("\nSegmentation Metrics")
    print("Mean IoU:", metrics["mean_iou"])
    print("Dice:", metrics["dice"])

    # ==============================
    # Save epoch model
    # ==============================
    torch.save(model.state_dict(), f"/kaggle/working/model_epoch_{epoch}.pth")

    # ==============================
    # Write logs
    # ==============================
    with open(LOG_PATH, "a") as f:
        f.write(f"\nEpoch {epoch+1}\n")
        f.write("----------------------------\n")
        f.write(f"Train Loss: {train_loss}\n")
        f.write("Detection Metrics\n")
        f.write(f"mAP: {current_map}\n")
        f.write("Segmentation Metrics\n")
        f.write(f"Mean IoU: {metrics['mean_iou']}\n")
        f.write(f"Dice: {metrics['dice']}\n")

    # ==============================
    # Save best model
    # ==============================
    if current_map > best_map:

        best_map = current_map
        best_epoch = epoch + 1

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'mAP': best_map
        }, WORKING_CHECKPOINT)

        print("\n Best model saved!")

        with open(LOG_PATH, "a") as f:
            f.write("Best model saved!\n")

    lr_scheduler.step()

# ==============================
# Final Summary
# ==============================
print("\n==============================")
print("Best Epoch:", best_epoch)
print("Best mAP:", best_map)

with open(LOG_PATH, "a") as f:
    f.write("\n==============================\n")
    f.write(f"Best Epoch: {best_epoch}\n")
    f.write(f"Best mAP: {best_map}\n")

In [ ]:
# def get_model_scratch(num_classes):

#     model = maskrcnn_resnet50_fpn(
#         weights=None,
#         weights_backbone=None
#     )

#     in_features = model.roi_heads.box_predictor.cls_score.in_features

#     model.roi_heads.box_predictor = FastRCNNPredictor(
#         in_features,
#         num_classes
#     )

#     in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels

#     model.roi_heads.mask_predictor = MaskRCNNPredictor(
#         in_features_mask,
#         256,
#         num_classes
#     )

#     return model

In [ ]:
# model_scratch = get_model_scratch(NUM_CLASSES)
# model_scratch.to(device)

In [ ]:
# optimizer = torch.optim.AdamW(
#     model_scratch.parameters(),
#     lr=1e-4
# )

In [ ]:
# best_map = 0
# best_epoch = 0

# EPOCHS = 40

# for epoch in range(EPOCHS):

#     print("\nEpoch", epoch+1)
#     print("----------------------------")

#     train_loss = train_one_epoch(model_scratch, train_loader, optimizer)

#     metrics = evaluate_metrics(model_scratch, val_loader)

#     current_map = metrics["mAP"]

#     print("Train Loss:", train_loss)

#     print("\nDetection Metrics")
#     print("mAP:", current_map)

#     print("\nClassification Metrics")
#     print("Precision:", metrics["precision"])
#     print("Recall:", metrics["recall"])
#     print("F1:", metrics["f1"])

#     print("\nSegmentation Metrics")
#     print("Mean IoU:", metrics["mean_iou"])
#     print("Dice:", metrics["dice"])

#     # Save best model
#     if current_map > best_map:

#         best_map = current_map
#         best_epoch = epoch + 1

#         torch.save({
#             'epoch': best_epoch,
#             'model_state_dict': model_scratch.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'mAP': best_map
#         }, "best_maskrcnn_scratch.pth")

#         print("\nBest model saved!")

# print("\n==============================")
# print("Best Epoch:", best_epoch)
# print("Best mAP:", best_map)